In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

import os
from dotenv import load_dotenv, find_dotenv

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_25664\2901061338.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\Code\AI\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# loader the text file

loader = TextLoader("../../data/sample.txt", encoding="utf8")
documents = loader.load()

print(f"Number of documents: {len(documents)}")
print(f"First document: {documents[0].page_content[:100]}...")
print(f"Last document: {documents[-1].page_content[:100]}...")

Number of documents: 1
First document: # 🦜️🔗 LangChain Comprehensive Knowledge Base & Practical Testing Guide

---

## 1. Overview and Core...
Last document: # 🦜️🔗 LangChain Comprehensive Knowledge Base & Practical Testing Guide

---

## 1. Overview and Core...


In [5]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

print(f"Number of documents before splitting: {len(documents)}")
print(f"Number of documents after splitting: {len(docs)}")


Created a chunk of size 1011, which is longer than the specified 1000


Number of documents before splitting: 1
Number of documents after splitting: 9


In [6]:
# third is embedding the documents 


hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
print(f"Using Hugging Face token: {hf_token[:4]}...{hf_token[-4:]}")

hf_embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)




Using Hugging Face token: hf_b...oTYL


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_25664\125159354.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8054.84it/s]


In [7]:
# fourth create the vector store
vector_store = FAISS.from_documents(docs, hf_embeddings)

print(f"Number of documents in vector store: {vector_store.index.ntotal}")
print(f"Vector store dimension: {vector_store.index.d}")
print(f"Vector store index type: {type(vector_store.index)}")


Number of documents in vector store: 9
Vector store dimension: 384
Vector store index type: <class 'faiss.swigfaiss.IndexFlatL2'>


In [8]:
## test querying the vector store
query = "What is the recommended chunk_size and chunk_overlap for general Q&A?"

results = vector_store.similarity_search(query, k=3) # k is the number of results to return
print(f"Query: {query}")
for i, result in enumerate(results):
    print(f"Result {i+1}: {result.page_content[:100]}...")

Query: What is the recommended chunk_size and chunk_overlap for general Q&A?
Result 1: Best Practices for Chunk Size & Overlap:
- For general Q&A with dense embeddings: `chunk_size = 500`...
Result 2: ---

## 3. Text Splitting and Chunking Strategies
Because LLMs and embedding models have strict cont...
Result 3: ---

## 6. Retrievers & Search Strategies
A Retriever is a high-level LangChain abstraction that acc...


In [9]:
# سؤال بدون ذكر كلمة chunk_size أو overlap
query = "How long should text segments be when preparing data for language models?"
results2 = vector_store.similarity_search(query, k=2)
print(f"Query: {query}")
print("*"*50)
print(f"Number of results: {len(results2)}")
print("*"*50)
print(results2[0].page_content)


Query: How long should text segments be when preparing data for language models?
**************************************************
Number of results: 2
**************************************************
Best Practices for Chunk Size & Overlap:
- For general Q&A with dense embeddings: `chunk_size = 500` to `1000` characters.
- Chunk overlap (`chunk_overlap = 100` to `200` characters) ensures critical semantic context is not severed across chunk boundaries.

---

## 4. Embeddings & Semantic Vector Space
An embedding model converts textual passages into high-dimensional numerical vectors (e.g., 384 dimensions for `all-MiniLM-L6-v2`, 1536 dimensions for `text-embedding-3-small`).

Embedding Models Categories:
- Local Hugging Face Models: Run 100% on local hardware via `sentence-transformers` without API costs.
- FastEmbed: An ultra-fast CPU-optimized embedding engine powered by ONNX Runtime.
- Cloud Providers: OpenAI, Cohere, Google Vertex AI, and Voyage AI.


In [ ]:
# save the vector store to disk
vector_store.save_local("vector_store")
